# 모델 실험 템플릿

팀 공통 실험 노트북입니다. **이 파일은 고치지 말고 복사해서** 씁니다.
각자 모델 하나(`yolo11m.pt` 또는 `yolo26m.pt`)를 맡아 옵티마이저·lr0 등을 조정합니다.

1. `notebooks/exp_<이름>.ipynb`로 복사합니다.
2. 설정 셀의 **내 실험** 부분만 바꿉니다. **팀 공통** 값은 모두 같아야 서로 점수를 비교할 수 있으므로 바꾸지 않습니다.
3. 순서
   1. 로컬(Mac)에서 `SMOKE = True`로 동작 확인 (epoch 1, 학습 사진 10%, wandb 끔)
   2. Colab GPU에서 `SMOKE = False`, `STAGE = "screen"`: fold 0만 학습해 설정을 고릅니다. 다른 설정과는 fold 0 점수끼리 비교합니다.
   3. 고른 설정은 `STAGE = "confirm"`: 5 fold를 모두 학습합니다. fold 0은 screen 결과를 그대로 씁니다. **모델 비교는 이 5 fold 평균·표준편차로** 합니다.
4. 실험 이름은 설정으로 자동으로 만듭니다 (예: `a_yolo26m_AdamW_lr0.001_cos`). 같은 설정을 다시 돌리면 끝난 fold는 건너뛰고, 끊긴 fold는 `last.pt`에서 이어 학습합니다.
5. 결과는 wandb 팀 프로젝트에 `<실험 이름>/<run>` 이름으로 쌓이고, config에 옵티마이저·lr0 등이 남습니다.
   - Colab: 커널 선택 → **Colab** → GPU 런타임. VS Code 탐색기에서 `.env`를 우클릭 → **Upload to Colab** (`DATA_URL`, `WANDB_API_KEY`, `WANDB_ENTITY`).
   - Colab은 `BRANCH`의 코드를 받아 돌리므로, 로컬에서 고친 `.py`는 push한 뒤 실행합니다.
6. 커밋할 때는 출력을 지웁니다.

## 1. 설정

실험마다 **내 실험**과 **실행** 부분만 바꿉니다. 팀 공통 값을 바꿔야 할 일이 생기면 팀에서 정한 뒤 템플릿을 고칩니다.

In [ ]:
# ===== 팀 공통: 바꾸지 않는다 (AGENTS.md 기본값) =====
LABEL = "v1"
SPLIT = "setup30fix_v1_s42"
EPOCHS = 100  # 기존 MuSGD 조건으로 100에폭 실험
IMGSZ = 640
SEED = 42
AUG = "on"  # models/train.py의 AUG_PRESETS
FRACTION = 1.0

# ===== 내 실험 =====
MEMBER = "JJAM99"  # 이름이나 이니셜. 실험 이름 맨 앞에 붙는다
MODEL = "yolo11m.pt"  # yolo11m.pt 또는 yolo26m.pt
# SGD, MuSGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp. auto는 lr0를 무시하므로 쓰지 않는다
OPTIMIZER = "MuSGD"
LR0 = None  # None이면 SGD 계열 0.01, 그 밖 0.001
COS_LR = False  # True면 학습률을 cosine 곡선으로 줄인다
BATCH = 16  # 팀 기본 16. GPU 메모리가 부족하면 줄인다
# 그 밖 train() 인자. 괄호 안은 ultralytics 기본값
# lrf(0.01), momentum(0.937), weight_decay(0.0005), warmup_epochs(3.0), patience(100)
TRAIN_KWARGS = {}  # 예: {"weight_decay": 0.001, "patience": 30}
TAG = ""  # 이름이 같아지는 실험을 구분할 짧은 꼬리표 (선택)

# ===== 실행 =====
SMOKE = False  # True면 동작 확인만. Colab GPU 학습은 False
STAGE = "screen"  # screen: fold 0만, confirm: 5 fold 전부
BRANCH = "exp/yolo11m_musgd"  # Colab에서 받을 브랜치

## 2. 환경

Colab이면 저장소를 받고 패키지를 설치한 뒤 결과 폴더를 Drive에 둡니다. 로컬에서는 프로젝트 루트를 import 경로에 넣기만 합니다.

In [ ]:
import importlib
import os
import re
import subprocess
import sys
import tempfile
import tomllib
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").is_dir()
RUNS_DIR = None


def dependency_snapshot(text: str) -> str:
    """한쪽이 비어 있는 추가 충돌만 임시 설치 사본에서 합친다. 원본은 수정하지 않는다."""
    pattern = re.compile(
        r"^<<<<<<<[^\n]*\n(.*?)^=======\n(.*?)^>>>>>>>[^\n]*\n?", re.M | re.S
    )

    def merge_addition(match: re.Match[str]) -> str:
        left, right = match.group(1), match.group(2)
        if left.strip() and right.strip():
            raise RuntimeError(
                "양쪽 내용이 다른 의존성 충돌입니다. 자동으로 선택하지 않습니다."
            )
        return left if left.strip() else right

    snapshot = pattern.sub(merge_addition, text)
    if any(
        line.startswith(("<<<<<<<", "=======", ">>>>>>>"))
        for line in snapshot.splitlines()
    ):
        raise RuntimeError("의존성 충돌 형식을 해석할 수 없습니다.")
    tomllib.loads(snapshot)
    return snapshot


if IN_COLAB:
    REPO_DIR = Path("/content/pill-bbox")
    if REPO_DIR.exists():
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True
        )
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True
        )
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "-q"], check=True
        )
    else:
        subprocess.run(
            [
                "git",
                "clone",
                "-q",
                "-b",
                BRANCH,
                "https://github.com/def-uk/pill-bbox.git",
                str(REPO_DIR),
            ],
            check=True,
        )
    os.chdir(REPO_DIR)
    uv_path = Path("~/.local/bin/uv").expanduser()
    if not uv_path.exists():
        installer = subprocess.run(
            ["curl", "-LsSf", "https://astral.sh/uv/install.sh"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout
        subprocess.run(["sh"], input=installer, text=True, check=True)
    os.environ["PATH"] = f"{uv_path.parent}:{os.environ['PATH']}"
    # 충돌 표시는 임시 디렉터리의 사본에서만 처리한다. 저장소 파일은 그대로 둔다.
    with tempfile.TemporaryDirectory(prefix="pill-dependencies-") as temporary_dir:
        dependency_dir = Path(temporary_dir)
        for filename in ("pyproject.toml", "uv.lock"):
            original = (REPO_DIR / filename).read_text(encoding="utf-8")
            snapshot = dependency_snapshot(original)
            if snapshot != original:
                print(f"{filename}: 한쪽 추가 충돌을 임시 설치 사본에서 처리합니다.")
            (dependency_dir / filename).write_text(snapshot, encoding="utf-8")
        requirements_path = dependency_dir / "requirements.txt"
        subprocess.run(
            [
                "uv",
                "export",
                "--project",
                str(dependency_dir),
                "--frozen",
                "--no-dev",
                "--no-hashes",
                "--no-emit-project",
                "-o",
                str(requirements_path),
            ],
            cwd=dependency_dir,
            check=True,
        )
        subprocess.run(
            [
                "uv",
                "pip",
                "install",
                "--python",
                sys.executable,
                "-r",
                str(requirements_path),
                "--extra-index-url",
                "https://download.pytorch.org/whl/cu126",
                "--index-strategy",
                "unsafe-best-match",
            ],
            cwd=dependency_dir,
            check=True,
        )
    importlib.invalidate_caches()
    from dotenv import load_dotenv

    load_dotenv("/content/.env")
    for name in ("DATA_URL", "WANDB_API_KEY", "WANDB_ENTITY"):
        if not os.getenv(name):
            try:
                from google.colab import userdata

                value = userdata.get(name)
                if value:
                    os.environ[name] = value
            except Exception:
                pass
    if not os.getenv("WANDB_API_KEY"):
        os.environ["WANDB_MODE"] = "disabled"
        print("WANDB_API_KEY가 없어 wandb를 끄고 진행합니다.")
    # 100에폭 결과가 런타임 종료 후에도 남도록 Drive 연결 실패 시 학습 전에 멈춘다.
    from google.colab import drive

    drive.mount("/content/drive")
    RUNS_DIR = Path("/content/drive/MyDrive/pill-bbox/runs")
    subprocess.run(["git", "branch", "--show-current"], check=True)
    subprocess.run(["git", "log", "--oneline", "-1"], check=True)
else:
    notebook_root = Path.cwd()
    if not (notebook_root / "notebooks" / "nb_data.py").is_file():
        notebook_root = notebook_root.parent
    REPO_DIR = notebook_root

# 프로젝트 루트와 notebooks 양쪽에서 실행할 수 있다.
for import_path in (REPO_DIR, REPO_DIR / "notebooks"):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
import nb_data  # noqa: E402, F401

## 3. 실험 준비

설정을 검사하고 실험 이름을 만듭니다. 같은 학습을 터미널에서 다시 돌리는 `model_main` 명령을 출력하고, 설정은 `<RUNS_DIR>/<exp>/notebook_config.json`에 남깁니다.

In [ ]:
import json
from datetime import datetime

from data_process.split import N_FOLDS
from models.train import default_lr0
from src.config import PROJECT_ROOT, get_device
from src.config import RUNS_DIR as DEFAULT_RUNS_DIR

# 가중치 이름(yolo11m.pt)은 현재 폴더에 받아지므로 루트로 옮겨 팀 공통 위치에 둔다
os.chdir(PROJECT_ROOT)
RUNS_DIR = RUNS_DIR or DEFAULT_RUNS_DIR
DEVICE = get_device()

if not MEMBER:
    raise ValueError("MEMBER에 이름이나 이니셜을 적으세요.")
if MODEL not in ["yolo11m.pt", "yolo26m.pt"]:
    raise ValueError(f"후보 모델은 yolo11m.pt, yolo26m.pt입니다: {MODEL}")
if OPTIMIZER == "auto":
    raise ValueError("auto는 lr0를 무시합니다. 옵티마이저를 직접 고르세요 (예: AdamW).")
if STAGE not in ["screen", "confirm"]:
    raise ValueError(f"STAGE는 screen 또는 confirm입니다: {STAGE}")
lr0 = default_lr0(OPTIMIZER) if LR0 is None else LR0

# 결과 폴더 이름에는 lr0 등이 없으므로, 내 실험 설정을 모두 실험 이름에 넣어 구분한다
parts = [MEMBER, Path(MODEL).stem, OPTIMIZER, f"lr{lr0:g}"]
parts += ["cos"] if COS_LR else []
parts += [] if BATCH == 16 else [f"b{BATCH}"]
parts += [
    f"{key}{value:g}" if isinstance(value, float) else f"{key}{value}"
    for key, value in sorted(TRAIN_KWARGS.items())
]
parts += [TAG] if TAG else []
exp_name = "_".join(parts)

# 동작 확인이면 짧은 값을 쓰고, 실험 이름에 시각을 붙여 이전 확인과 겹치지 않게 한다
if SMOKE:
    exp, epochs, fraction, folds = (
        f"smoke_{exp_name}_{datetime.now():%m%d_%H%M}",
        1,
        0.1,
        [0],
    )
    os.environ["WANDB_MODE"] = "disabled"
else:
    if DEVICE != "cuda":
        raise RuntimeError(
            f"본 학습은 GPU에서 합니다(지금 {DEVICE}). 동작 확인은 SMOKE = True."
        )
    exp = f"{exp_name}_test{EPOCHS}"
    epochs, fraction = EPOCHS, FRACTION
    folds = [0] if STAGE == "screen" else list(range(N_FOLDS))

# 같은 학습을 터미널에서 다시 돌릴 수 있게 model_main 명령으로 남긴다
cmd = ["uv run python -m models.model_main", "--model", MODEL, "--label", LABEL]
cmd += [
    "--split",
    SPLIT,
    "--folds",
    *map(str, folds),
    "--epochs",
    str(epochs),
    "--exp",
    exp,
]
cmd += ["--imgsz", str(IMGSZ), "--batch", str(BATCH), "--seed", str(SEED)]
cmd += [
    "--fraction",
    str(fraction),
    "--aug",
    AUG,
    "--optimizer",
    OPTIMIZER,
    "--lr0",
    f"{lr0:g}",
]
cmd += ["--cos-lr"] if COS_LR else []
cmd += [] if RUNS_DIR == DEFAULT_RUNS_DIR else ["--runs-dir", str(RUNS_DIR)]
cmd += ["--resume"]
command = " ".join(cmd)

config = {
    "exp": exp,
    "member": MEMBER,
    "stage": STAGE,
    "smoke": SMOKE,
    "device": DEVICE,
    "model": MODEL,
    "label": LABEL,
    "split": SPLIT,
    "folds": folds,
    "epochs": epochs,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "fraction": fraction,
    "aug": AUG,
    "optimizer": OPTIMIZER,
    "lr0": lr0,
    "cos_lr": COS_LR,
    "train_kwargs": TRAIN_KWARGS,
    "command": command,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
exp_dir = RUNS_DIR / exp
exp_dir.mkdir(parents=True, exist_ok=True)
config_path = exp_dir / "notebook_config.json"
if config_path.is_file():
    previous_config = json.loads(config_path.read_text(encoding="utf-8"))
    ignored_keys = {"created_at", "stage", "folds", "command", "device"}
    changed_keys = [
        key
        for key, value in config.items()
        if key not in ignored_keys and previous_config.get(key) != value
    ]
    if changed_keys:
        raise ValueError(
            f"기존 실험과 설정이 다릅니다: {changed_keys}. TAG로 새 실험을 만드세요."
        )
config_path.write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"실험 {exp} ({STAGE}, fold {folds}), 장치 {DEVICE}")
print(f"결과 {exp_dir}")
print(command)
if TRAIN_KWARGS:
    print(f"추가 train() 인자 (CLI에 없음): {TRAIN_KWARGS}")

## 4. 데이터

데이터는 `data_process.data_main`으로만 만듭니다. fold마다 train/val 사진 수입니다.

In [ ]:
# 다른 PC의 절대 경로가 남았다면 정식 파이프라인으로 한 번만 다시 만든다.
import yaml
from nb_data import ensure_pipeline, load_folds

from data_process.data_yaml import data_yaml_path

ensure_pipeline(LABEL, [SPLIT])

for preparation_attempt in range(2):
    stale_paths = []
    for fold in folds:
        dataset = yaml.safe_load(
            data_yaml_path(LABEL, SPLIT, fold).read_text(encoding="utf-8")
        )
        split_images = {}
        for subset in ("train", "val"):
            list_path = Path(dataset[subset])
            if not list_path.is_file():
                stale_paths.append(str(list_path))
                continue
            paths = [
                Path(line.strip())
                for line in list_path.read_text().splitlines()
                if line.strip()
            ]
            if not paths:
                raise ValueError(f"fold {fold} {subset} 이미지 목록이 비어 있습니다.")
            stale_paths.extend(str(image) for image in paths if not image.is_file())
            split_images[subset] = {image.resolve() for image in paths}
        if split_images.get("train", set()) & split_images.get("val", set()):
            raise ValueError(f"fold {fold}: train/val에 같은 이미지가 있습니다.")
    if not stale_paths:
        break
    if preparation_attempt:
        raise FileNotFoundError(
            f"파이프라인 재실행 후에도 이미지 경로가 없습니다: {stale_paths[:3]}"
        )
    print("현재 PC와 맞지 않는 경로를 발견해 data_main으로 다시 준비합니다.")
    ensure_pipeline(LABEL, [SPLIT], rerun=True)
load_folds(SPLIT).groupby(["fold", "split"]).size().unstack()

In [ ]:
# 평가·비교 함수: 보조 파일 없이 이 노트북 안에서 함께 사용한다.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from matplotlib.ticker import MaxNLocator
from ultralytics import YOLO
from ultralytics.data.utils import img2label_paths
from ultralytics.utils.metrics import box_iou
from ultralytics.utils.ops import xywh2xyxy

from data_process.data_yaml import data_yaml_path
from models.evaluate import map75_95
from models.train import MAIN_METRIC_KEY
from src.config import get_device


def matched_iou(
    model: YOLO, data: Path, imgsz: int, batch: int
) -> dict[str, float | int]:
    """같은 클래스·IoU 0.5 이상인 예측과 정답을 일대일 매칭해 평균과 개수를 반환한다."""
    dataset = yaml.safe_load(data.read_text(encoding="utf-8"))
    matched = []
    target_count = 0
    prediction_count = 0
    for prediction in model.predict(
        source=dataset["val"],
        imgsz=imgsz,
        batch=batch,
        device=get_device(),
        conf=0.001,
        iou=0.7,
        stream=True,
        verbose=False,
        save=False,
    ):
        label_path = Path(img2label_paths([prediction.path])[0])
        targets = np.array(
            [
                list(map(float, line.split()))
                for line in label_path.read_text().splitlines()
                if line.strip()
            ],
            dtype=np.float32,
        ).reshape(-1, 5)
        target_count += len(targets)
        prediction_count += len(prediction.boxes)
        if not len(targets) or not len(prediction.boxes):
            continue
        height, width = prediction.orig_shape
        boxes = xywh2xyxy(torch.from_numpy(targets[:, 1:].copy()))
        boxes *= torch.tensor([width, height, width, height])
        overlaps = box_iou(boxes, prediction.boxes.xyxy.cpu()).numpy()
        same_class = targets[:, 0, None] == prediction.boxes.cls.cpu().numpy()[None, :]
        target_indices, prediction_indices = np.where(same_class & (overlaps >= 0.5))
        order = np.argsort(-overlaps[target_indices, prediction_indices], kind="stable")
        used_targets, used_predictions = set(), set()
        for index in order:
            target_index = int(target_indices[index])
            prediction_index = int(prediction_indices[index])
            if target_index in used_targets or prediction_index in used_predictions:
                continue
            used_targets.add(target_index)
            used_predictions.add(prediction_index)
            matched.append(float(overlaps[target_index, prediction_index]))
    return {
        "Mean IoU": float(np.mean(matched)) if matched else float("nan"),
        "Matched": len(matched),
        "Targets": target_count,
        "Unmatched targets": target_count - len(matched),
        "Predictions": prediction_count,
    }


def find_smoke_runs(reference: Path) -> list[Path]:
    """같은 모델·분할·fold와 주요 설정으로 완료된 5에폭 이하 스모크를 찾는다."""
    settings = yaml.safe_load((reference / "args.yaml").read_text())
    keys = (
        "model",
        "optimizer",
        "lr0",
        "cos_lr",
        "batch",
        "imgsz",
        "seed",
        "degrees",
        "fliplr",
        "flipud",
        "hsv_h",
        "hsv_s",
        "hsv_v",
        "mosaic",
        "momentum",
        "weight_decay",
        "warmup_epochs",
        "patience",
        "lrf",
        "freeze",
    )
    candidates = []
    for path in reference.parent.parent.glob(f"smoke_*/{reference.name}"):
        args_path = path / "args.yaml"
        history_path = path / "results.csv"
        if not args_path.is_file() or not history_path.is_file():
            continue
        args = yaml.safe_load(args_path.read_text())
        if not (path / "weights" / "best.pt").is_file():
            continue
        if not 0 < args["epochs"] <= 5 or args.get("fraction") != 0.1:
            continue
        if any(args.get(key) != settings.get(key) for key in keys):
            continue
        history = pd.read_csv(history_path)
        history.columns = history.columns.str.strip()
        if len(history) and history["epoch"].max() >= args["epochs"]:
            candidates.append(path)
    return sorted(
        candidates, key=lambda path: (path / "weights" / "best.pt").stat().st_mtime
    )


def compare_runs(
    runs: dict[str, Path],
    label: str,
    split: str,
    fold: int,
    imgsz: int,
    batch: int,
    output_dir: Path,
) -> pd.DataFrame:
    """각 run의 best.pt를 같은 val 전체로 평가해 비교표·최종 지표 그림·학습 곡선을 저장한다."""
    output_dir.mkdir(parents=True, exist_ok=True)
    data = data_yaml_path(label, split, fold)
    rows = []
    histories = {}
    for name, run in runs.items():
        settings = yaml.safe_load((run / "args.yaml").read_text())
        history = pd.read_csv(run / "results.csv")
        history.columns = history.columns.str.strip()
        histories[name] = history
        weights = run / "weights" / "best.pt"
        print(f"{name}: {weights}")
        model = YOLO(str(weights))
        result = model.val(
            data=str(data),
            imgsz=imgsz,
            batch=batch,
            device=get_device(),
            conf=0.001,
            iou=0.7,
            plots=False,
            project=str(output_dir / "val"),
            name=name,
            exist_ok=True,
        )
        rows.append(
            {
                "Experiment": name,
                "Epochs": int(history["epoch"].max()),
                "Fraction": settings["fraction"],
                "mAP75-95": map75_95(result.box),
                "Recall": float(result.box.mr),
                "Precision": float(result.box.mp),
                **matched_iou(model, data, imgsz, batch),
                "Weights": str(weights),
            }
        )
    summary = pd.DataFrame(rows).set_index("Experiment")
    columns = [
        "Epochs",
        "Fraction",
        "mAP75-95",
        "Mean IoU",
        "Recall",
        "Precision",
        "Matched",
        "Targets",
        "Unmatched targets",
        "Predictions",
        "Weights",
    ]
    summary = summary[columns]
    summary.to_csv(output_dir / "comparison.csv", encoding="utf-8")
    _plot_comparison(summary, histories, output_dir)
    return summary


def _plot_comparison(
    summary: pd.DataFrame,
    histories: dict[str, pd.DataFrame],
    output_dir: Path,
) -> None:
    """최종 지표의 막대그래프와 에폭별 꺾은선 그래프를 표시하고 저장한다."""
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    for ax, metric in zip(
        axes, ["mAP75-95", "Mean IoU", "Recall", "Precision"], strict=True
    ):
        values = summary[metric]
        bars = ax.bar(
            summary.index, values, color=[f"C{i}" for i in range(len(summary))]
        )
        ax.bar_label(bars, labels=[f"{value:.4f}" for value in values], padding=3)
        ax.set(title=metric, ylim=(0, 1.1))
        ax.tick_params(axis="x", labelrotation=20)
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Best checkpoint: same validation set (IoU: matched objects only)")
    fig.tight_layout()
    fig.savefig(output_dir / "comparison_metrics.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for ax, (key, title) in zip(
        axes.flat,
        [
            (MAIN_METRIC_KEY, "Validation mAP75-95"),
            ("metrics/recall(B)", "Validation recall"),
            ("metrics/precision(B)", "Validation precision"),
            ("train/cls_loss", "Classification loss (solid: train / dashed: val)"),
        ],
        strict=True,
    ):
        for index, (name, history) in enumerate(histories.items()):
            ax.plot(
                history["epoch"],
                history[key],
                marker=".",
                color=f"C{index}",
                label=name,
            )
            if key == "train/cls_loss":
                ax.plot(
                    history["epoch"], history["val/cls_loss"], "--", color=f"C{index}"
                )
        ax.set(title=title, xlabel="Epoch")
        if key != "train/cls_loss":
            ax.set_ylim(0, 1)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle("Training history (see Epochs / Fraction in comparison.csv)")
    fig.tight_layout()
    fig.savefig(output_dir / "comparison_history.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 5. 학습과 평가

`best.pt`는 검증 mAP75-95가 가장 높은 epoch입니다. 런타임이 끊기면 위에서부터 다시 실행하면 이어서 학습합니다.

In [ ]:
import traceback
from datetime import datetime

from models.evaluate import MAIN_METRIC, evaluate_fold, summarize_exp
from models.train import run_dir, train

# 기존 30에폭 체크포인트와 섞이지 않게 epochs가 포함된 실험 폴더에서만 재개한다.
for fold in folds:
    save_dir = run_dir(MODEL, SPLIT, fold, exp, AUG, OPTIMIZER, RUNS_DIR)
    save_dir.mkdir(parents=True, exist_ok=True)
    status_path = save_dir / "notebook_status.json"
    stage = "preflight"
    try:
        args_path = save_dir / "args.yaml"
        if args_path.is_file():
            previous_args = yaml.safe_load(args_path.read_text(encoding="utf-8"))
            expected_args = {
                "epochs": epochs,
                "imgsz": IMGSZ,
                "batch": BATCH,
                "seed": SEED,
                "fraction": fraction,
                "optimizer": OPTIMIZER,
                "lr0": lr0,
                "cos_lr": COS_LR,
                **TRAIN_KWARGS,
            }
            changed = [
                key
                for key, value in expected_args.items()
                if previous_args.get(key) != value
            ]
            if changed:
                raise ValueError(
                    f"저장된 학습 설정과 다릅니다: {changed}. TAG를 바꿔 실행하세요."
                )
        stage = "train"
        status_path.write_text(
            json.dumps(
                {
                    "stage": stage,
                    "updated_at": datetime.now().isoformat(),
                    "command": command,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        save_dir = train(
            MODEL,
            LABEL,
            SPLIT,
            fold,
            epochs=epochs,
            exp=exp,
            imgsz=IMGSZ,
            batch=BATCH,
            seed=SEED,
            fraction=fraction,
            aug=AUG,
            optimizer=OPTIMIZER,
            lr0=lr0,
            cos_lr=COS_LR,
            runs_dir=RUNS_DIR,
            resume=True,
            **TRAIN_KWARGS,
        )
        stage = "evaluate"
        status_path.write_text(
            json.dumps(
                {
                    "stage": stage,
                    "updated_at": datetime.now().isoformat(),
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        _, summary = evaluate_fold(
            save_dir / "weights" / "best.pt",
            LABEL,
            SPLIT,
            fold,
            exp=exp,
            imgsz=IMGSZ,
            batch=BATCH,
            runs_dir=RUNS_DIR,
        )
        status_path.write_text(
            json.dumps(
                {
                    "stage": "complete",
                    "updated_at": datetime.now().isoformat(),
                    "metric": MAIN_METRIC,
                    "score": float(summary[MAIN_METRIC]),
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        print(f"{save_dir.name}: {MAIN_METRIC} {summary[MAIN_METRIC]:.4f}")
    except (Exception, KeyboardInterrupt) as error:
        status_path.write_text(
            json.dumps(
                {
                    "stage": stage,
                    "updated_at": datetime.now().isoformat(),
                    "error_type": type(error).__name__,
                    "error": str(error),
                    "traceback": traceback.format_exc(),
                    "command": command,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )
        print(f"{stage} 단계가 중단됐습니다. 오류 기록: {status_path}")
        print(
            "원인을 확인한 뒤 이 셀을 다시 실행하면 같은 실험의 last.pt에서 재개합니다."
        )
        print("검증만 실패했다면 완료된 학습은 건너뛰고 검증을 다시 실행합니다.")
        raise

## 6. 결과 요약

다른 사람·다른 설정과 비교할 때 screen은 fold 0끼리, 모델 비교는 confirm의 5 fold 평균·표준편차로 합니다.

In [ ]:
# fold 평균·표준편차. <exp>/results/summary.csv에도 저장된다. screen은 fold 하나라 std가 비어 있다.
summary = summarize_exp(exp, RUNS_DIR)
cols = [("n_folds", "count"), (MAIN_METRIC, "mean"), (MAIN_METRIC, "std")]
cols += [("map50", "mean"), ("map50_95", "mean")]
summary[cols].round(4)

## 7. 학습 곡선

fold별 매 epoch 학습·검증 loss(윗줄)와 검증 mAP(아랫줄)입니다. 같은 그림이 run 폴더의 `results.png`로도 저장됩니다.

In [ ]:
import pandas as pd

METRIC_KEYS = ["metrics/mAP50(B)", "metrics/mAP50-95(B)", MAIN_METRIC_KEY]

# ultralytics가 매 epoch 쓰는 results.csv를 fold별로 읽는다 (run 폴더 이름 끝이 _fold<k>)
histories = {}
for path in sorted(exp_dir.glob("*/results.csv")):
    history = pd.read_csv(path)
    history.columns = history.columns.str.strip()
    histories[path.parent.name.rsplit("_", 1)[-1]] = history

if not histories:
    print(f"{exp_dir}에 results.csv가 없습니다. 학습 셀을 먼저 실행하세요.")
else:
    first = next(iter(histories.values()))
    loss_names = [
        c.removeprefix("train/") for c in first.columns if c.startswith("train/")
    ]
    n_cols = max(len(loss_names), len(METRIC_KEYS))
    fig, axes = plt.subplots(2, n_cols, figsize=(4.5 * n_cols, 7), squeeze=False)
    for i, (fold, history) in enumerate(histories.items()):
        color, epoch = f"C{i}", history["epoch"]
        # 윗줄: loss. 실선은 train, 점선은 val. val이 오르는데 train만 내려가면 과적합이다
        for ax, name in zip(axes[0], loss_names, strict=False):
            ax.plot(
                epoch,
                history[f"train/{name}"],
                color=color,
                marker=".",
                label=f"{fold} train",
            )
            ax.plot(
                epoch,
                history[f"val/{name}"],
                color=color,
                linestyle="--",
                label=f"{fold} val",
            )
            ax.set_title(name)
        # 아랫줄: 검증 mAP. 별은 best.pt를 고른 epoch (mAP75-95 최고)
        for ax, key in zip(axes[1], METRIC_KEYS, strict=False):
            ax.plot(epoch, history[key], color=color, marker=".", label=fold)
            ax.set_title(key.removeprefix("metrics/").removesuffix("(B)"))
        best = history[MAIN_METRIC_KEY].idxmax()
        axes[1, len(METRIC_KEYS) - 1].plot(
            epoch[best],
            history.loc[best, MAIN_METRIC_KEY],
            "*",
            color=color,
            markersize=14,
        )
    for ax in axes.ravel():
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    for ax in [*axes[0, len(loss_names) :], *axes[1, len(METRIC_KEYS) :]]:
        ax.set_visible(False)
    axes[0, 0].legend(fontsize=7)
    axes[1, 0].legend(fontsize=7)
    fig.suptitle(exp)
    fig.tight_layout()
    plt.show()

In [ ]:
# 8. AP: best.pt를 val 전체로 평가한다. 이 셀은 학습 없이 검증 추론을 실행한다.
# AP는 클래스별 값, mAP는 클래스 평균이다. 주 지표 mAP75-95는 그대로 유지한다.
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

from data_process.data_yaml import load_class_map
from models.evaluate import ap75_95

metric_runs = []
class_map = load_class_map(LABEL)
for weights in sorted(exp_dir.glob("*/weights/best.pt")):
    metric_fold = int(weights.parent.parent.name.rsplit("_fold", 1)[1])
    if metric_fold not in folds:
        continue
    metric_model = YOLO(str(weights))
    metric_data = data_yaml_path(LABEL, SPLIT, metric_fold)
    metric_result = metric_model.val(
        data=str(metric_data),
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        conf=0.001,
        iou=0.7,
        plots=False,
        project=str(exp_dir / "results" / "metric_val"),
        name=weights.parent.parent.name,
        exist_ok=True,
    )
    metric_runs.append(
        (weights.parent.parent.name, weights, metric_data, metric_result)
    )
    del metric_model

if not metric_runs:
    raise FileNotFoundError(
        "선택한 fold의 best.pt가 없습니다. 학습 셀을 먼저 실행하세요."
    )

ap_rows = []
for run_name, _, _, result in metric_runs:
    box = result.box
    class_ap = pd.DataFrame(
        {
            "yolo_idx": box.ap_class_index,
            "AP50": box.ap50,
            "AP50-95": box.ap,
            "AP75-95": ap75_95(box.all_ap),
        }
    ).merge(class_map, on="yolo_idx", how="left")
    print(run_name, "클래스별 AP (val에 정답이 있는 클래스)")
    display(class_ap.round(4))
    ap_rows.append(
        {
            "run": run_name,
            "mAP50": box.map50,
            "mAP50-95": box.map,
            "mAP75-95": map75_95(box),
        }
    )
display(pd.DataFrame(ap_rows).set_index("run").round(4))

In [ ]:
# 9. IoU: 같은 클래스·IoU >= 0.5인 예측과 정답을 일대일 매칭한다.
# 30에폭 비교 셀과 같은 계산 함수를 사용한다. 미검출은 평균에서 빠진다.

if "matched_iou" not in globals():
    raise RuntimeError("데이터 셀 아래의 평가·비교 함수 셀을 먼저 실행하세요.")

iou_rows = []
for run_name, metric_weights, metric_data, _ in metric_runs:
    metric_model = YOLO(str(metric_weights))
    iou_result = matched_iou(metric_model, metric_data, IMGSZ, BATCH)
    iou_rows.append(
        {
            "run": run_name,
            "mean_matched_IoU@0.5": iou_result["Mean IoU"],
            "matched": iou_result["Matched"],
            "targets": iou_result["Targets"],
            "predictions": iou_result["Predictions"],
            "unmatched_targets": iou_result["Unmatched targets"],
        }
    )
    del metric_model
display(pd.DataFrame(iou_rows).set_index("run").round(4))

In [ ]:
# 10. 재현율: TP / (TP + FN). val 클래스별 값과 클래스 평균을 표시한다.
# Ultralytics의 IoU=0.5, 평균 F1이 최대인 confidence 지점의 값이다.
# IoU 셀의 conf=0.001에서 집계한 매칭 비율과는 기준이 다르다.
recall_rows = []
for run_name, _, _, result in metric_runs:
    class_recall = pd.DataFrame(
        {
            "yolo_idx": result.box.ap_class_index,
            "recall": result.box.r,
        }
    ).merge(class_map, on="yolo_idx", how="left")
    print(run_name)
    display(class_recall.round(4))
    recall_rows.append({"run": run_name, "mean_recall": result.box.mr})
display(pd.DataFrame(recall_rows).set_index("run").round(4))

In [ ]:
# 11. 정밀도: TP / (TP + FP). 재현율과 같은 confidence 지점의 값이다.
precision_rows = []
for run_name, _, _, result in metric_runs:
    class_precision = pd.DataFrame(
        {
            "yolo_idx": result.box.ap_class_index,
            "precision": result.box.p,
        }
    ).merge(class_map, on="yolo_idx", how="left")
    print(run_name)
    display(class_precision.round(4))
    precision_rows.append({"run": run_name, "mean_precision": result.box.mp})
display(pd.DataFrame(precision_rows).set_index("run").round(4))

## 실험 기록

- **실험 이름 / 단계**: (설정 셀 출력의 이름, screen 또는 confirm)
- **이 설정을 고른 이유**:
- **명령**: 실험 준비 셀이 출력한 `uv run python -m models.model_main ...`
- **결과**: screen이면 fold 0 mAP75-95, confirm이면 5 fold 평균 ± 표준편차
- **결론 / 다음 실험**: 

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from models.train import MAIN_METRIC_KEY

# 12. 앞의 평가 셀 결과를 모아 네 가지 지표를 한 번에 확인한다.
# AP는 주 지표인 클래스 평균 mAP75-95로 표시한다.
# IoU는 IoU >= 0.5로 매칭된 예측만의 평균이다. 미검출 수를 함께 확인한다.
metric_summary = (
    pd.DataFrame(ap_rows)
    .set_index("run")[["mAP75-95"]]
    .join(
        pd.DataFrame(iou_rows).set_index("run")[
            ["mean_matched_IoU@0.5", "matched", "unmatched_targets"]
        ],
        validate="one_to_one",
    )
    .join(
        pd.DataFrame(recall_rows).set_index("run")[["mean_recall"]],
        validate="one_to_one",
    )
    .join(
        pd.DataFrame(precision_rows).set_index("run")[["mean_precision"]],
        validate="one_to_one",
    )
)
metric_summary = metric_summary[
    [
        "mAP75-95",
        "mean_matched_IoU@0.5",
        "mean_recall",
        "mean_precision",
        "matched",
        "unmatched_targets",
    ]
].rename(columns={"mean_recall": "Recall", "mean_precision": "Precision"})
display(metric_summary.round(4))

# 저장된 에폭별 기록을 읽어 최종 평가표와 학습 추이를 함께 보여 준다.
# 평균 매칭 IoU는 best.pt에서만 계산하므로 에폭별 곡선을 만들지 않는다.

history_paths = sorted(exp_dir.glob("*/results.csv"))
history_paths = [
    path
    for path in history_paths
    if int(path.parent.name.rsplit("_fold", 1)[1]) in folds
]
if not history_paths:
    print("에폭별 results.csv가 없습니다. 학습 셀을 먼저 실행하세요.")
else:
    for history_path in history_paths:
        history = pd.read_csv(history_path)
        history.columns = history.columns.str.strip()
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        for loss_name in ("box_loss", "cls_loss", "dfl_loss"):
            for split_name, style in (("train", "-"), ("val", "--")):
                key = f"{split_name}/{loss_name}"
                if key in history:
                    axes[0].plot(
                        history["epoch"],
                        history[key],
                        linestyle=style,
                        marker=".",
                        label=key,
                    )
        for key, label in (
            (MAIN_METRIC_KEY, "mAP75-95"),
            ("metrics/precision(B)", "Precision"),
            ("metrics/recall(B)", "Recall"),
        ):
            if key in history:
                axes[1].plot(history["epoch"], history[key], marker="o", label=label)
        axes[0].set(title="Training / validation loss", ylabel="Loss")
        axes[1].set(title="Validation metrics per epoch", ylabel="Score", ylim=(0, 1))
        for ax in axes:
            ax.set_xlabel("Epoch")
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
            ax.grid(alpha=0.3)
            ax.legend()
        fig.suptitle(f"{exp} / {history_path.parent.name}")
        fig.tight_layout()
        figure_path = history_path.parent / "experiment_history.png"
        fig.savefig(figure_path, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    print("평균 매칭 IoU는 best.pt의 최종 평가표에서 확인하세요.")

In [ ]:
# 13. 완료된 현재 실험과 기존 스모크·30에폭 결과를 비교한다. 추가 학습은 하지 않는다.
SMOKE_RUN_DIR = None  # 자동 검색 대신 지정할 때는 weights/가 들어 있는 run 폴더

if "compare_runs" not in globals():
    raise RuntimeError("데이터 아래의 평가·비교 함수 셀을 먼저 실행하세요.")
comparison_dir = exp_dir / "results" / "smoke_comparison"
for comparison_fold in folds:
    test_save_dir = run_dir(
        MODEL, SPLIT, comparison_fold, exp, AUG, OPTIMIZER, RUNS_DIR
    )
    if not (test_save_dir / "weights" / "best.pt").is_file():
        raise FileNotFoundError(f"학습 결과가 없습니다: {test_save_dir}")
    smoke_candidates = find_smoke_runs(test_save_dir)
    if SMOKE_RUN_DIR is not None:
        smoke_save_dir = Path(SMOKE_RUN_DIR)
        if smoke_save_dir.resolve() not in {
            path.resolve() for path in smoke_candidates
        }:
            raise ValueError("같은 설정·fold의 완료된 스모크 run을 지정하세요.")
    else:
        smoke_save_dir = smoke_candidates[-1] if smoke_candidates else None
    comparison_runs = {}
    if smoke_save_dir is not None:
        comparison_runs["Smoke"] = smoke_save_dir
    previous_run = run_dir(
        MODEL,
        SPLIT,
        comparison_fold,
        f"{exp_name}_test30",
        AUG,
        OPTIMIZER,
        RUNS_DIR,
    )
    if EPOCHS != 30 and (previous_run / "weights" / "best.pt").is_file():
        previous_args = yaml.safe_load((previous_run / "args.yaml").read_text())
        current_args = yaml.safe_load((test_save_dir / "args.yaml").read_text())
        compared_keys = (
            "model",
            "data",
            "imgsz",
            "batch",
            "seed",
            "fraction",
            "optimizer",
            "lr0",
            "cos_lr",
            "degrees",
            "fliplr",
            "flipud",
            "hsv_h",
            "hsv_s",
            "hsv_v",
            "mosaic",
            "momentum",
            "weight_decay",
            "warmup_epochs",
            "patience",
            "lrf",
            "freeze",
        )
        if all(
            previous_args.get(key) == current_args.get(key) for key in compared_keys
        ):
            comparison_runs["Test 30"] = previous_run
        else:
            print("30에폭 결과는 설정이 달라 비교에서 제외합니다.")
    comparison_runs[f"Test {epochs}"] = test_save_dir
    comparison = compare_runs(
        comparison_runs,
        LABEL,
        SPLIT,
        comparison_fold,
        IMGSZ,
        BATCH,
        comparison_dir / f"fold_{comparison_fold}",
    )
    display(comparison.drop(columns="Weights").round(4))
print("동일 fold의 val 전체 평가입니다. 스모크는 학습 데이터량도 다릅니다.")
print(f"비교 결과: {comparison_dir}")

In [ ]:
# 14. 알약 이미지: 같은 검증 사진의 정답(왼쪽)과 best.pt 예측(오른쪽)을 비교한다.
# 학습 없이 실행한다. 표시용 confidence는 평가 지표 계산과 별개다.
import random

from matplotlib.patches import Rectangle

PREVIEW_COUNT = 4
PREVIEW_CONF = 0.25
preview_dir = exp_dir / "results" / "pill_previews"
preview_weights = [
    weights
    for weights in sorted(exp_dir.glob("*/weights/best.pt"))
    if int(weights.parent.parent.name.rsplit("_fold", 1)[1]) in folds
]
if not preview_weights:
    raise FileNotFoundError(
        "선택한 fold의 best.pt가 없습니다. 학습 결과 경로를 확인하세요."
    )

preview_dir.mkdir(parents=True, exist_ok=True)
for weights in preview_weights:
    preview_fold = int(weights.parent.parent.name.rsplit("_fold", 1)[1])
    preview_data = yaml.safe_load(
        data_yaml_path(LABEL, SPLIT, preview_fold).read_text(encoding="utf-8")
    )
    val_images = sorted(
        line.strip()
        for line in Path(preview_data["val"]).read_text(encoding="utf-8").splitlines()
        if line.strip()
    )
    if not val_images:
        print(f"fold {preview_fold}: 검증 이미지가 없습니다.")
        continue
    preview_images = random.Random(SEED).sample(
        val_images, min(PREVIEW_COUNT, len(val_images))
    )
    preview_model = YOLO(str(weights))
    for image_index, prediction in enumerate(
        preview_model.predict(
            source=preview_images,
            imgsz=IMGSZ,
            device=get_device(),
            conf=PREVIEW_CONF,
            stream=True,
            save=False,
            verbose=False,
        )
    ):
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(prediction.orig_img[:, :, ::-1])
        height, width = prediction.orig_shape
        label_path = Path(img2label_paths([prediction.path])[0])
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            class_id, center_x, center_y, box_width, box_height = map(
                float, line.split()
            )
            left = (center_x - box_width / 2) * width
            top = (center_y - box_height / 2) * height
            axes[0].add_patch(
                Rectangle(
                    (left, top),
                    box_width * width,
                    box_height * height,
                    fill=False,
                    edgecolor="lime",
                    linewidth=2,
                )
            )
            axes[0].text(
                left,
                top,
                str(int(class_id)),
                color="black",
                bbox={"facecolor": "lime", "alpha": 0.8, "pad": 1},
            )
        axes[0].set_title("Ground truth (class ID)")
        axes[1].imshow(prediction.plot(labels=False)[:, :, ::-1])
        axes[1].set_title(
            f"Prediction: {len(prediction.boxes)} pills (conf >= {PREVIEW_CONF})"
        )
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"{weights.parent.parent.name} / {Path(prediction.path).name}")
        fig.tight_layout()
        figure_path = (
            preview_dir / f"{weights.parent.parent.name}_{image_index:02d}.png"
        )
        fig.savefig(figure_path, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
print(f"알약 이미지 저장 위치: {preview_dir}")

In [ ]:
# 15. 중복 검출 진단: 선택한 fold의 val 전체를 검사한다. 학습·평가 설정은 바꾸지 않는다.
# IoU, 작은 박스 포함률, 같은 정답에 겹친 예측을 함께 검사한다.
# 기준을 넘은 쌍은 중복 '의심'이며, 실제로 겹친 서로 다른 알약일 수도 있다.
from itertools import combinations

DUPLICATE_CONF = PREVIEW_CONF
DUPLICATE_IOU = 0.5
DUPLICATE_CONTAINMENT = 0.8
DUPLICATE_GT_IOU = 0.5
DUPLICATE_IMAGE_LIMIT = None  # None: val 전체, 정수: fold별 앞쪽 N장만 검사
DUPLICATE_PLOT_LIMIT = 12  # 전체 검사 중 의심 이미지 저장·표시 최대 개수

duplicate_dir = exp_dir / "results" / "duplicate_diagnostics"
duplicate_dir.mkdir(parents=True, exist_ok=True)
detection_rows, duplicate_rows, image_rows = [], [], []
plotted_count = 0
if not preview_weights:
    raise FileNotFoundError("14번 셀에서 선택한 fold의 best.pt를 먼저 확인하세요.")

for weights in preview_weights:
    run_name = weights.parent.parent.name
    diagnostic_fold = int(run_name.rsplit("_fold", 1)[1])
    diagnostic_data = yaml.safe_load(
        data_yaml_path(LABEL, SPLIT, diagnostic_fold).read_text(encoding="utf-8")
    )
    diagnostic_images = sorted(
        line.strip()
        for line in Path(diagnostic_data["val"])
        .read_text(encoding="utf-8")
        .splitlines()
        if line.strip()
    )
    if DUPLICATE_IMAGE_LIMIT is not None:
        diagnostic_images = diagnostic_images[:DUPLICATE_IMAGE_LIMIT]
    if not diagnostic_images:
        continue
    diagnostic_model = YOLO(str(weights))
    # 기존 클래스별 NMS를 유지한 상태에서 남은 중복을 검사한다.
    for prediction in diagnostic_model.predict(
        source=diagnostic_images,
        imgsz=IMGSZ,
        device=get_device(),
        conf=DUPLICATE_CONF,
        agnostic_nms=False,
        stream=True,
        save=False,
        verbose=False,
    ):
        image_path = str(prediction.path)
        boxes = prediction.boxes.xyxy.cpu().numpy()
        classes = prediction.boxes.cls.cpu().numpy().astype(int)
        confidences = prediction.boxes.conf.cpu().numpy()
        label_path = Path(img2label_paths([image_path])[0])
        targets = np.array(
            [
                list(map(float, line.split()))
                for line in label_path.read_text(encoding="utf-8").splitlines()
                if line.strip()
            ],
            dtype=np.float32,
        ).reshape(-1, 5)
        height, width = prediction.orig_shape
        target_boxes = xywh2xyxy(torch.from_numpy(targets[:, 1:].copy()))
        target_boxes *= torch.tensor([width, height, width, height])
        overlaps = box_iou(torch.from_numpy(boxes), torch.from_numpy(boxes)).numpy()
        target_overlaps = box_iou(torch.from_numpy(boxes), target_boxes).numpy()
        areas = np.prod(np.maximum(boxes[:, 2:] - boxes[:, :2], 0), axis=1)
        for index, (box, class_id, confidence) in enumerate(
            zip(boxes, classes, confidences, strict=True)
        ):
            detection_rows.append(
                {
                    "run": run_name,
                    "image": image_path,
                    "box_id": index,
                    "class_id": int(class_id),
                    "class_name": prediction.names[int(class_id)],
                    "confidence": float(confidence),
                    **dict(zip(("x1", "y1", "x2", "y2"), map(float, box), strict=True)),
                }
            )
        suspicious_ids = set()
        pair_count = 0
        for first, second in combinations(range(len(boxes)), 2):
            intersection_size = np.maximum(
                np.minimum(boxes[first, 2:], boxes[second, 2:])
                - np.maximum(boxes[first, :2], boxes[second, :2]),
                0,
            )
            containment = float(
                np.prod(intersection_size)
                / max(
                    min(areas[first], areas[second]),
                    1e-9,
                )
            )
            shared_targets = np.flatnonzero(
                (target_overlaps[first] >= DUPLICATE_GT_IOU)
                & (target_overlaps[second] >= DUPLICATE_GT_IOU)
            )
            pair_iou = float(overlaps[first, second])
            reasons = []
            if pair_iou >= DUPLICATE_IOU:
                reasons.append("high_iou")
            if containment >= DUPLICATE_CONTAINMENT:
                reasons.append("contained_box")
            if len(shared_targets):
                reasons.append("shared_ground_truth")
            if not reasons:
                continue
            pair_count += 1
            suspicious_ids.update((first, second))
            duplicate_rows.append(
                {
                    "run": run_name,
                    "image": image_path,
                    "box_a": first,
                    "box_b": second,
                    "class_a": int(classes[first]),
                    "class_b": int(classes[second]),
                    "name_a": prediction.names[int(classes[first])],
                    "name_b": prediction.names[int(classes[second])],
                    "conf_a": float(confidences[first]),
                    "conf_b": float(confidences[second]),
                    "same_class": bool(classes[first] == classes[second]),
                    "iou": pair_iou,
                    "containment": containment,
                    "shared_gt_ids": ";".join(map(str, shared_targets)),
                    "reasons": ";".join(reasons),
                }
            )
        image_rows.append(
            {
                "run": run_name,
                "image": image_path,
                "targets": len(targets),
                "predictions": len(boxes),
                "suspicious_pairs": pair_count,
                "suspicious_boxes": len(suspicious_ids),
            }
        )
        if not suspicious_ids or plotted_count >= DUPLICATE_PLOT_LIMIT:
            continue
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(prediction.orig_img[:, :, ::-1])
        for index, box in enumerate(boxes):
            left, top, right, bottom = box
            color = f"C{index % 10}"
            ax.add_patch(
                Rectangle(
                    (left, top),
                    right - left,
                    bottom - top,
                    fill=False,
                    edgecolor=color,
                    linewidth=2,
                    linestyle="--" if index in suspicious_ids else "-",
                )
            )
            # 박스가 거의 같아도 번호·클래스·점수가 가려지지 않도록 범례를 따로 둔다.
            ax.annotate(
                f"#{index}",
                (left, top),
                xytext=(5, -12 - 16 * (index % 3)),
                textcoords="offset points",
                color=color,
                arrowprops={"arrowstyle": "-", "color": color},
            )
            ax.plot(
                [],
                [],
                color=color,
                label=(
                    f"#{index}: class {classes[index]}, conf {confidences[index]:.3f}"
                    + (" [suspect]" if index in suspicious_ids else "")
                ),
            )
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
        ax.set_title(
            f"{run_name} / {Path(image_path).name}: {pair_count} suspect pairs"
        )
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(
            duplicate_dir / f"{run_name}_{plotted_count:03d}.png",
            dpi=150,
            bbox_inches="tight",
        )
        plt.show()
        plt.close(fig)
        plotted_count += 1

# 결과가 없어도 헤더를 저장하고, 재실행하면 CSV를 갱신한다.
detection_table = pd.DataFrame(
    detection_rows,
    columns=[
        "run",
        "image",
        "box_id",
        "class_id",
        "class_name",
        "confidence",
        "x1",
        "y1",
        "x2",
        "y2",
    ],
)
duplicate_table = pd.DataFrame(
    duplicate_rows,
    columns=[
        "run",
        "image",
        "box_a",
        "box_b",
        "class_a",
        "class_b",
        "name_a",
        "name_b",
        "conf_a",
        "conf_b",
        "same_class",
        "iou",
        "containment",
        "shared_gt_ids",
        "reasons",
    ],
)
image_table = pd.DataFrame(
    image_rows,
    columns=[
        "run",
        "image",
        "targets",
        "predictions",
        "suspicious_pairs",
        "suspicious_boxes",
    ],
)
for filename, table in (
    ("detections.csv", detection_table),
    ("duplicate_pairs.csv", duplicate_table),
    ("image_summary.csv", image_table),
):
    table.to_csv(duplicate_dir / filename, index=False, encoding="utf-8-sig")
(diagnostic_settings_path := duplicate_dir / "settings.json").write_text(
    json.dumps(
        {
            "confidence": DUPLICATE_CONF,
            "pair_iou": DUPLICATE_IOU,
            "containment": DUPLICATE_CONTAINMENT,
            "gt_iou": DUPLICATE_GT_IOU,
            "image_limit": DUPLICATE_IMAGE_LIMIT,
            "plot_limit": DUPLICATE_PLOT_LIMIT,
            "weights": [str(weights) for weights in preview_weights],
            "agnostic_nms": False,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"검사 {len(image_table)}장 / 중복 의심 {len(duplicate_table)}쌍")
print(
    "의심 쌍은 중복 알약 개수가 아닙니다. 같은 알약의 여러 박스가 여러 쌍으로 집계됩니다."
)
if not duplicate_table.empty:
    display(duplicate_table.sort_values("iou", ascending=False).head(30).round(4))
    suspicious_images = image_table[image_table["suspicious_pairs"] > 0]
    display(suspicious_images.head(30))
else:
    print("현재 confidence·검사 기준에서 중복 의심 예측이 없습니다.")
print(f"전체 좌표·클래스·점수·의심 쌍 CSV와 그림: {duplicate_dir}")

In [ ]:
# 16. 어떤 약을 못 잡았나: 100에폭 실험의 best.pt로 val 전체 실패 사례를 조사한다.
# 표시용 conf=0.25, 정답 판정 IoU=0.5. 공식 AP·재현율과 판정 기준이 다르다.
# 위치·종류 오류의 원인 구분은 일대일 탐욕 매칭에 기반한 진단용 추정이다.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from matplotlib.patches import Rectangle
from nb_data import load_fold_labels
from ultralytics import YOLO
from ultralytics.data.utils import img2label_paths
from ultralytics.utils.metrics import box_iou
from ultralytics.utils.ops import xywh2xyxy

from data_process.data_yaml import data_yaml_path, load_class_map
from src.config import get_device

ERROR_CONF = 0.25
ERROR_IOU = 0.5
ERROR_STRICT_IOU = 0.75
ERROR_NEAR_IOU = 0.1
ERROR_PLOT_LIMIT = 12
ERROR_WEIGHTS = (
    "best.pt"  # 100번째 epoch 자체를 보려면 last.pt. 기본은 검증 점수 최적 모델
)


def diagnose_matches(
    overlaps: np.ndarray,
    target_classes: np.ndarray,
    prediction_classes: np.ndarray,
    match_iou: float,
    strict_iou: float,
    near_iou: float,
) -> list[dict]:
    """정답 우선 일대일 매칭으로 미검출·종류 혼동·위치 오차·추가 예측을 나눈다."""
    used_targets, used_predictions, records = set(), set(), []
    same_class = target_classes[:, None] == prediction_classes[None, :]
    for phase in ("same_class", "wrong_class", "poor_box"):
        eligible = overlaps >= (near_iou if phase == "poor_box" else match_iou)
        if phase == "same_class":
            eligible &= same_class
        elif phase == "wrong_class":
            eligible &= ~same_class
        target_ids, prediction_ids = np.where(eligible)
        order = np.argsort(-overlaps[target_ids, prediction_ids], kind="stable")
        for pair_index in order:
            target_id, prediction_id = (
                int(target_ids[pair_index]),
                int(prediction_ids[pair_index]),
            )
            if target_id in used_targets or prediction_id in used_predictions:
                continue
            used_targets.add(target_id)
            used_predictions.add(prediction_id)
            overlap = float(overlaps[target_id, prediction_id])
            status = phase
            if phase == "same_class":
                status = "correct" if overlap >= strict_iou else "loose_box"
            records.append(
                {
                    "record_type": "target",
                    "target_id": target_id,
                    "prediction_id": prediction_id,
                    "status": status,
                    "iou": overlap,
                }
            )
    for target_id in range(len(target_classes)):
        if target_id not in used_targets:
            records.append(
                {
                    "record_type": "target",
                    "target_id": target_id,
                    "prediction_id": -1,
                    "status": "missed",
                    "iou": 0.0,
                }
            )
    for prediction_id in range(len(prediction_classes)):
        if prediction_id in used_predictions:
            continue
        target_id = (
            int(overlaps[:, prediction_id].argmax()) if len(target_classes) else -1
        )
        overlap = float(overlaps[target_id, prediction_id]) if target_id >= 0 else 0.0
        records.append(
            {
                "record_type": "extra_prediction",
                "target_id": target_id,
                "prediction_id": prediction_id,
                "status": "duplicate_suspect"
                if overlap >= match_iou
                else "false_positive",
                "iou": overlap,
            }
        )
    return records


def plot_failure_image(
    image: np.ndarray,
    targets: np.ndarray,
    predictions: np.ndarray,
    records: list[dict],
    title: str,
    destination: Path,
) -> None:
    """정답과 예측을 번호·오류별 색으로 나란히 표시하고 저장한다."""
    colors = {
        "correct": "limegreen",
        "loose_box": "gold",
        "wrong_class": "magenta",
        "poor_box": "darkorange",
        "missed": "red",
        "duplicate_suspect": "cyan",
        "false_positive": "deepskyblue",
    }
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, boxes, side in zip(
        axes, (targets, predictions), ("target", "prediction"), strict=True
    ):
        ax.imshow(image[:, :, ::-1])
        for index, (left, top, right, bottom) in enumerate(boxes):
            record = next(
                row
                for row in records
                if row[f"{side}_id"] == index
                and (side == "prediction" or row["record_type"] == "target")
            )
            color = colors[record["status"]]
            ax.add_patch(
                Rectangle(
                    (left, top),
                    right - left,
                    bottom - top,
                    fill=False,
                    edgecolor=color,
                    linewidth=2,
                )
            )
            ax.annotate(
                f"{side[0].upper()}{index}: {record['status']}",
                (left, top),
                xytext=(0, -8 - (index % 3) * 13),
                textcoords="offset points",
                color=color,
                fontsize=8,
                bbox={"facecolor": "black", "alpha": 0.65, "pad": 1},
                arrowprops={"arrowstyle": "-", "color": color},
            )
        ax.set_title(
            "Ground truth (T: target ID)"
            if side == "target"
            else "Prediction (P: box ID)"
        )
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(destination, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


if not 0 <= ERROR_NEAR_IOU < ERROR_IOU <= ERROR_STRICT_IOU <= 1:
    raise ValueError("IoU 기준은 0 <= near < match <= strict <= 1이어야 합니다.")
error_weights = [
    path
    for path in sorted(exp_dir.glob(f"*/weights/{ERROR_WEIGHTS}"))
    if int(path.parent.parent.name.rsplit("_fold", 1)[1]) in folds
]
if not error_weights:
    raise FileNotFoundError(f"선택한 실험에 {ERROR_WEIGHTS}가 없습니다: {exp_dir}")
error_dir = exp_dir / "results" / "failure_analysis" / Path(ERROR_WEIGHTS).stem
error_dir.mkdir(parents=True, exist_ok=True)
error_rows, class_tables, image_records = [], [], []
class_info = load_class_map(LABEL).set_index("yolo_idx")
true_positive_statuses = {"correct", "loose_box"}

for weights in error_weights:
    run_name = weights.parent.parent.name
    error_fold = int(run_name.rsplit("_fold", 1)[1])
    args = yaml.safe_load(
        (weights.parent.parent / "args.yaml").read_text(encoding="utf-8")
    )
    if args["epochs"] != 100:
        raise ValueError(
            f"100에폭으로 설정한 실험이 아닙니다: {run_name} ({args['epochs']})"
        )
    data = yaml.safe_load(
        data_yaml_path(LABEL, SPLIT, error_fold).read_text(encoding="utf-8")
    )
    image_list = [
        line.strip()
        for line in Path(data["val"]).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if not image_list:
        raise ValueError(f"fold {error_fold}의 검증 이미지가 없습니다.")
    model = YOLO(str(weights))
    run_records = []
    for prediction in model.predict(
        source=image_list,
        imgsz=IMGSZ,
        conf=ERROR_CONF,
        iou=0.7,
        device=get_device(),
        agnostic_nms=False,
        stream=True,
        save=False,
        verbose=False,
    ):
        label_path = Path(img2label_paths([prediction.path])[0])
        labels = np.array(
            [
                list(map(float, line.split()))
                for line in label_path.read_text(encoding="utf-8").splitlines()
                if line.strip()
            ],
            dtype=np.float32,
        ).reshape(-1, 5)
        height, width = prediction.orig_shape
        target_tensor = xywh2xyxy(torch.from_numpy(labels[:, 1:].copy()))
        target_tensor *= torch.tensor([width, height, width, height])
        target_boxes = target_tensor.numpy()
        predicted_boxes = prediction.boxes.xyxy.cpu().numpy()
        target_classes = labels[:, 0].astype(int)
        predicted_classes = prediction.boxes.cls.cpu().numpy().astype(int)
        confidences = prediction.boxes.conf.cpu().numpy()
        overlaps = box_iou(target_tensor, torch.from_numpy(predicted_boxes)).numpy()
        records = diagnose_matches(
            overlaps,
            target_classes,
            predicted_classes,
            ERROR_IOU,
            ERROR_STRICT_IOU,
            ERROR_NEAR_IOU,
        )
        for record in records:
            target_id, prediction_id = record["target_id"], record["prediction_id"]
            target_class = int(target_classes[target_id]) if target_id >= 0 else None
            predicted_class = (
                int(predicted_classes[prediction_id]) if prediction_id >= 0 else None
            )
            row = {
                "run": run_name,
                "fold": error_fold,
                "image": str(prediction.path),
                **record,
                "true_class": target_class,
                "predicted_class": predicted_class,
                "true_name": class_info.loc[target_class, "dl_name"]
                if target_class is not None
                else None,
                "predicted_name": class_info.loc[predicted_class, "dl_name"]
                if predicted_class is not None
                else None,
                "confidence": float(confidences[prediction_id])
                if prediction_id >= 0
                else None,
                "target_xyxy": json.dumps(target_boxes[target_id].tolist())
                if target_id >= 0
                else None,
                "prediction_xyxy": json.dumps(predicted_boxes[prediction_id].tolist())
                if prediction_id >= 0
                else None,
            }
            error_rows.append(row)
            run_records.append(row)
        missed_count = sum(
            row["status"] in {"missed", "wrong_class", "poor_box"} for row in records
        )
        extra_count = sum(row["record_type"] == "extra_prediction" for row in records)
        loose_count = sum(row["status"] == "loose_box" for row in records)
        image_records.append(
            {
                "run": run_name,
                "image": str(prediction.path),
                "targets": len(labels),
                "predictions": len(predicted_boxes),
                "fn_at_iou50": missed_count,
                "extra_predictions": extra_count,
                "loose_boxes": loose_count,
            }
        )
        # fold마다 처음 발견한 실패 이미지만 제한해서 표시한다. 전체 사례는 CSV에 남긴다.
        saved_count = sum(
            row["run"] == run_name and "plot" in row for row in image_records
        )
        if (
            missed_count or extra_count or loose_count
        ) and saved_count < ERROR_PLOT_LIMIT:
            destination = error_dir / f"{run_name}_{saved_count:03d}.png"
            plot_failure_image(
                prediction.orig_img,
                target_boxes,
                predicted_boxes,
                records,
                f"{run_name} / {Path(prediction.path).name}",
                destination,
            )
            image_records[-1]["plot"] = str(destination)
            display(
                pd.DataFrame(
                    [row for row in run_records if row["image"] == str(prediction.path)]
                )[
                    [
                        "record_type",
                        "target_id",
                        "prediction_id",
                        "status",
                        "true_name",
                        "predicted_name",
                        "confidence",
                        "iou",
                    ]
                ].round(4)
            )
    records_table = pd.DataFrame(run_records)
    fold_labels = load_fold_labels(SPLIT, error_fold)
    summary_rows = []
    for class_id, info in class_info.iterrows():
        if records_table.empty:
            targets = detections = records_table
        else:
            targets = records_table[
                (records_table["record_type"] == "target")
                & (records_table["true_class"] == class_id)
            ]
            detections = records_table[records_table["predicted_class"] == class_id]
        tp = (
            int(targets["status"].isin(true_positive_statuses).sum())
            if len(targets)
            else 0
        )
        fp = len(detections) - tp
        counts = (
            targets["status"].value_counts() if len(targets) else pd.Series(dtype=int)
        )
        summary_rows.append(
            {
                "run": run_name,
                "yolo_idx": class_id,
                "category_id": info["category_id"],
                "dl_name": info["dl_name"],
                "train_targets": int(
                    (
                        (fold_labels["category_id"] == info["category_id"])
                        & (fold_labels["split"] == "train")
                    ).sum()
                ),
                "val_targets": len(targets),
                "tp": tp,
                "fn": len(targets) - tp,
                "fp": fp,
                "recall_at_conf": tp / len(targets) if len(targets) else np.nan,
                "precision_at_conf": tp / len(detections)
                if len(detections)
                else np.nan,
                "missed": int(counts.get("missed", 0)),
                "wrong_class": int(counts.get("wrong_class", 0)),
                "poor_box_suspect": int(counts.get("poor_box", 0)),
                "loose_box_iou50_75": int(counts.get("loose_box", 0)),
            }
        )
    class_table = pd.DataFrame(summary_rows)
    ap_path = exp_dir / "results" / f"{run_name}_per_class.csv"
    # best.pt를 검사할 때만 같은 run의 기존 AP를 결합한다.
    if ERROR_WEIGHTS == "best.pt" and ap_path.is_file():
        ap_table = pd.read_csv(ap_path)[["yolo_idx", "ap75_95"]]
        class_table = class_table.merge(
            ap_table, on="yolo_idx", how="left", validate="one_to_one"
        )
    class_tables.append(class_table)
    del model

failure_details = pd.DataFrame(
    error_rows,
    columns=[
        "run",
        "fold",
        "image",
        "record_type",
        "target_id",
        "prediction_id",
        "status",
        "iou",
        "true_class",
        "predicted_class",
        "true_name",
        "predicted_name",
        "confidence",
        "target_xyxy",
        "prediction_xyxy",
    ],
)
class_failures = pd.concat(class_tables, ignore_index=True)
image_failures = pd.DataFrame(image_records)
confusion_pairs = (
    failure_details[failure_details["status"] == "wrong_class"]
    .groupby(["run", "true_class", "true_name", "predicted_class", "predicted_name"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)
for filename, table in (
    ("class_failures.csv", class_failures),
    ("failure_details.csv", failure_details),
    ("image_failures.csv", image_failures),
    ("confusion_pairs.csv", confusion_pairs),
):
    table.to_csv(error_dir / filename, index=False, encoding="utf-8-sig")
(error_dir / "settings.json").write_text(
    json.dumps(
        {
            "weights": [str(weights) for weights in error_weights],
            "confidence": ERROR_CONF,
            "match_iou": ERROR_IOU,
            "strict_iou": ERROR_STRICT_IOU,
            "near_iou": ERROR_NEAR_IOU,
            "nms_iou": 0.7,
            "agnostic_nms": False,
            "imgsz": IMGSZ,
            "label": LABEL,
            "split": SPLIT,
            "folds": folds,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("놓치거나 종류를 틀린 약: fn 내림차순, 같은 개수면 재현율이 낮은 순서")
display(
    class_failures[class_failures["val_targets"] > 0]
    .sort_values(
        ["fn", "recall_at_conf"],
        ascending=[False, True],
    )
    .head(20)
    .round(4)
)
print("추가로 잘못 예측한 약: fp가 많은 순서")
display(
    class_failures[class_failures["fp"] > 0]
    .sort_values("fp", ascending=False)
    .head(20)
    .round(4)
)
print("종류는 맞지만 박스가 덜 정확한 약 (IoU 0.5 이상·0.75 미만)")
display(
    class_failures[class_failures["loose_box_iou50_75"] > 0]
    .sort_values(
        "loose_box_iou50_75",
        ascending=False,
    )
    .head(20)
    .round(4)
)
print("실패가 많은 사진: 미검출·종류 혼동·위치 오차, 추가 예측 순서")
display(
    image_failures.sort_values(
        ["fn_at_iou50", "extra_predictions", "loose_boxes"],
        ascending=False,
    ).head(20)
)
print("어떤 약을 어떤 약으로 혼동했는지 (IoU >= 0.5인 일대일 대응)")
display(confusion_pairs.head(20))
print(
    "val_targets=0이면 이 fold에서 평가할 수 없습니다. 적은 표본의 재현율도 함께 주의해 보세요."
)
print("missed: 대응 예측 없음 / wrong_class: 종류 혼동 / poor_box: 위치 오류 의심")
print(
    "loose_box: 종류는 맞고 IoU 0.5 이상·0.75 미만 / duplicate_suspect: 추가 중복 의심"
)
print("별도 고정 confidence 진단입니다. 공식 AP와 동일한 지표가 아닙니다.")
print(f"상세 CSV·실패 이미지 저장 위치: {error_dir}")


In [ ]:
from models.evaluate import make_submission
from models.train import run_dir

SUBMIT_FOLD = folds[0]  # 제출에 쓸 fold의 best.pt. screen은 fold 0뿐이다
# 이 점수 미만의 박스는 버린다. 대회는 mAP@[0.75:0.95] 채점이라 낮게 둬도 점수가 떨어지지 않는다
SUBMIT_CONF = 0.001

weights = (
    run_dir(MODEL, SPLIT, SUBMIT_FOLD, exp, AUG, OPTIMIZER, RUNS_DIR)
    / "weights"
    / "best.pt"
)
if not weights.is_file():
    raise FileNotFoundError(f"{weights}가 없습니다. 학습 셀을 먼저 실행하세요.")
submission, submission_path = make_submission(
    weights, LABEL, exp, IMGSZ, SUBMIT_CONF, RUNS_DIR
)
print(f"사진 {submission['image_id'].nunique()}장, 박스 {len(submission)}개")
print(f"저장 {submission_path}")
submission.head()